# StegaGuard Phase 6 — Kaggle GPU training

The Phase 6 plan calls for one real GPU training/evaluation run; it recommends Colab, but Kaggle GPU is an alternative. **Do not rerun Phase 5.** This notebook prepares BOSSbase, runs the required 100-cover CPU smoke test, trains StegNet-Lite on the full cover-disjoint splits, and packages the model and results.

## Before running
1. Create a Kaggle Dataset containing the supplied `stegaguard-phase6-kaggle-pipeline.zip` (keep that exact filename).
2. In this notebook, use **Add Input** to attach that pipeline dataset and the raw BOSSbase 1.01 dataset (`lunan0/bossbase-1-01`). The BOSSbase images stay on Kaggle; no local dataset download is needed.
3. In **Notebook options**, enable a GPU accelerator. Dataset access is through attached Kaggle inputs; Internet is only needed if the runtime is missing a Python dependency. Run cells from top to bottom.

Kaggle stores outputs under `/kaggle/working/stegaguard-phase6`. Download `stegaguard-phase6-model-results.zip` from the notebook's Output panel when the final cell succeeds. The 100-cover smoke run is only a pipeline check, not a benchmark.

In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/stegaguard-phase6")

# Find the pipeline directory or zip file anywhere recursively under /kaggle/input (handling Kaggle's nested datasets/ structure)
pipeline_candidates = [
    p
    for p in INPUT_ROOT.rglob("*")
    if p.is_dir() and "stegaguard-phase6-kaggle-pipeline" in p.name.casefold()
]
if pipeline_candidates:
    BACKEND = pipeline_candidates[0]
    print("Found attached pipeline dataset directory:", BACKEND)
else:
    pipeline_zips = list(INPUT_ROOT.rglob("stegaguard-phase6-kaggle-pipeline.zip"))
    if len(pipeline_zips) != 1:
        raise FileNotFoundError(
            "Attach the Kaggle input dataset containing stegaguard-phase6-kaggle-pipeline; "
            f"found {len(pipeline_zips)} matching archives/folders."
        )
    PIPELINE_ROOT = WORK_ROOT / "stegaguard-phase6-pipeline"
    if PIPELINE_ROOT.exists():
        shutil.rmtree(PIPELINE_ROOT)
    PIPELINE_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(pipeline_zips[0]) as archive:
        archive.extractall(PIPELINE_ROOT)
    BACKEND = PIPELINE_ROOT
    print("Extracted pipeline zip to:", BACKEND)

sys.path.insert(0, str(BACKEND))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
print("Pipeline backend root:", BACKEND)
print("Working outputs:", WORK_ROOT)

In [ ]:
import importlib.util

import torch

required_modules = {
    "numpy": "numpy<2",
    "scipy": "scipy",
    "PIL": "pillow",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
    "matplotlib": "matplotlib",
}
missing_packages = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]
if missing_packages:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Find BOSSbase and prepare the reproducible splits

This accepts either an attached prepared Phase 4 dataset (containing `dataset_card.json` and `splits.json`) or the attached raw BOSSbase images. If automatic discovery finds multiple BOSSbase inputs, `RAW_DATA` is discovered automatically via `.pgm` file lookup.

In [ ]:
PREPARED_OUTPUT = WORK_ROOT / "bossbase-prepared"
prepared_candidates = [
    path.parent
    for path in INPUT_ROOT.rglob("dataset_card.json")
    if (path.parent / "splits.json").is_file()
]
if len(prepared_candidates) == 1:
    DATASET = prepared_candidates[0]
    print("Using attached prepared dataset:", DATASET)
else:
    pgm_files = list(INPUT_ROOT.rglob("*.pgm"))
    if not pgm_files:
        raise FileNotFoundError(
            "Attach the raw BOSSbase 1.01 dataset or an attached prepared dataset containing .pgm files."
        )
    RAW_DATA = pgm_files[0].parent
    print("Found raw BOSSbase image folder:", RAW_DATA)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "scripts.prepare_dataset",
            "--src",
            str(RAW_DATA),
            "--out",
            str(PREPARED_OUTPUT),
            "--size",
            "256",
            "--seed",
            "1337",
            "--origin",
            "BOSSbase 1.01 from Kaggle dataset lunan0/bossbase-1-01",
        ],
        cwd=BACKEND,
        check=True,
    )
    DATASET = PREPARED_OUTPUT
dataset_card = json.loads((DATASET / "dataset_card.json").read_text(encoding="utf-8"))
print("Dataset origin:", dataset_card["origin"])
print("Prepared images:", dataset_card["image_count"])
print("Splits:", dataset_card["splits"])

## Required CPU smoke test

Run this first. It uses 100 covers total, two CPU epochs, and one condition. Its metrics are not model benchmarks.

In [ ]:
SMOKE_ROOT = WORK_ROOT / "smoke"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.train_cnn",
        "--dataset",
        str(DATASET),
        "--models-out",
        str(SMOKE_ROOT / "models"),
        "--results-out",
        str(SMOKE_ROOT / "results"),
        "--fixed-out",
        str(SMOKE_ROOT / "fixed"),
        "--device",
        "cpu",
        "--max-total-covers",
        "100",
        "--epochs",
        "2",
        "--patience",
        "2",
        "--batch-pairs",
        "4",
        "--payloads",
        "0.4",
        "--methods",
        "lsb_replace",
        "--modes",
        "keyed",
        "--bootstrap-resamples",
        "10",
        "--latency-samples",
        "4",
    ],
    cwd=BACKEND,
    check=True,
)
print("CPU smoke test completed. These scores are not benchmark results.")

## Full GPU training and held-out evaluation

Run only after the smoke test succeeds. This follows the Phase 6 defaults: up to 40 epochs, 8-epoch early stopping patience, all five payload rates, both embedding methods and modes, validation-only checkpoint selection/calibration, and test-only reporting. Expect a long Kaggle session; keep the notebook session active until the command finishes.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before full CNN training.")
FULL_ROOT = WORK_ROOT / "full"
subprocess.run(
    [
        sys.executable,
        "-m",
        "scripts.train_cnn",
        "--dataset",
        str(DATASET),
        "--models-out",
        str(FULL_ROOT / "models"),
        "--results-out",
        str(FULL_ROOT / "results"),
        "--fixed-out",
        str(FULL_ROOT / "fixed"),
        "--device",
        "cuda",
        "--epochs",
        "40",
        "--patience",
        "8",
        "--batch-pairs",
        "32",
        "--num-workers",
        "2",
        "--seed",
        "1337",
        "--bootstrap-resamples",
        "1000",
    ],
    cwd=BACKEND,
    check=True,
)
print("Checkpoint:", FULL_ROOT / "models" / "cnn_v1.pt")
print("Metadata:", FULL_ROOT / "models" / "cnn_v1.json")
print("Metrics:", FULL_ROOT / "results" / "cnn_metrics.json")
print("Curves:", FULL_ROOT / "results" / "plots" / "cnn_training_curves.png")

## Package artifacts for download

Only the full-run checkpoint, metadata, test metrics, and curve are included. Keep this archive for the repository's Phase 6 review; do not present smoke metrics as BOSSbase benchmark results.

In [ ]:
ARTIFACT_ZIP = WORK_ROOT / "stegaguard-phase6-model-results.zip"
artifact_files = [
    FULL_ROOT / "models" / "cnn_v1.pt",
    FULL_ROOT / "models" / "cnn_v1.json",
    FULL_ROOT / "results" / "cnn_metrics.json",
    FULL_ROOT / "results" / "plots" / "cnn_training_curves.png",
]
missing_files = [str(path) for path in artifact_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f"Full-run outputs are missing: {missing_files}")
with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_files:
        archive.write(path, path.relative_to(FULL_ROOT))
print("Download from Kaggle notebook outputs:", ARTIFACT_ZIP)